In [0]:
# --------------------------------------------------------------------------------------------------
# SCRIPT: 2_membros_frentes_transform
# LOCAL: 2_silver/src/1_atlas_frentes/
# OBJETIVO: Converter os tipos e estrutura referente os dados da tabela Bronze_Membros_Frentes
# --------------------------------------------------------------------------------------------------

from pyspark.sql.functions import col

# Carregar tabelas 
df_membros_bronze = spark.table("bronze_membros_frentes")
df_deputados_silver = spark.table("silver_deputados")

# Transformar a Bronze de membros
df_membros_limpo = df_membros_bronze.select(
    col("id").alias("id_frente"),
    col("`deputado_.id`").alias("id_deputado"),
    col("`deputado_.nome`").alias("nome_deputado"),
    col("`deputado_.siglaUf`").alias("uf")
)

df_silver_completo = df_membros_limpo.join(
    df_deputados_silver, 
    df_membros_limpo.id_deputado == df_deputados_silver.id, 
    "left"
).select(
    col("id_frente"),
    col("id_deputado"),
    col("nome_deputado"),
    col("siglaPartido").alias("partido"), 
    col("uf")
).distinct()

# Salvar na Silver
df_silver_completo.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_membros_frentes")

display(df_silver_completo)